# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook guides you through exploring the FAIR² dataset, using the [mlcroissant](https://github.com/mlcommons/croissant) library for loading, inspecting, and preprocessing the data. All dataset entities are referenced using their unique `@id`.

### Dataset Source
The dataset's metadata and structure are described using the Croissant schema at this URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed in your environment
!pip install mlcroissant

## 1. Data Loading

We first load the dataset's metadata. The mlcroissant API allows seamless access to Croissant datasets, including schema, field definitions, and, when available, data records.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Access the metadata object (as per mlcroissant's API)
metadata = dataset.metadata
print('Dataset Name:')
print(metadata.name)
print('\nDescription:')
print(metadata.description)

# Optionally display additional summary fields
print('\nIdentifier:', getattr(metadata, 'identifier', '[Not available]'))
print('Published:', getattr(metadata, 'datePublished', '[Not available]'))

## 2. Data Overview

We inspect available record sets (`cr:RecordSet`/`@id`), their field `@id`s, and, if present, column structures. This allows us to reference entities explicitly by their `@id` as recommended in Croissant datasets.

**Note:** Record sets, fields, and columns are referenced by their exact `@id` as per the Croissant schema.

In [ ]:
# List all record sets in the dataset and their fields

record_sets = dataset.record_sets
if not record_sets:
    print('No record sets are defined in this dataset (recordSet is empty in metadata).')
else:
    for rs in record_sets:
        print(f'Record Set: {rs.id}')
        print(f'  Name: {rs.name}')
        print(f'  Description: {getattr(rs, "description", "")}')
        print('  Fields:')
        for fld in rs.fields:
            print(f'    Field @id: {fld.id}  (name: {getattr(fld, "name", "")})')
        print('  Columns:')
        for col in getattr(rs, 'columns', []):
            print(f'    Column @id: {col.id} (name: {getattr(col, "name", "")})')
        print('-' * 40)

# Save record set @ids for use in later cells
record_set_ids = [rs.id for rs in record_sets] if record_sets else []
# For demonstration, print the found @ids:
print('Available record set @ids:', record_set_ids)

## 3. Data Extraction

We extract all records for each available record set (by their `@id`). Data are loaded into DataFrames, making it easier to manipulate them. If the dataset includes no record sets or sample data, this cell will demonstrate that as well.

In [ ]:
# Load data records from all record sets (by their @id)
dataframes = {}

if not record_set_ids:
    print('No record sets defined; no tabular data to extract.')
else:
    for rs_id in record_set_ids:
        # All records in this record set as dictionaries
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded Record Set: {rs_id}")
        print(f"  Columns: {df.columns.tolist()}")
        print(f"  First few rows:")
        display(df.head(3))
        print('-'*40)

# If there is at least one record set with data, select it for following cells
if dataframes:
    selected_record_set_id = list(dataframes.keys())[0]
    print(f'Example record set selected for analysis: {selected_record_set_id}')
else:
    selected_record_set_id = None

## 4. Exploratory Data Analysis (EDA)

Here we demonstrate standard data processing steps, such as filtering numerics, normalization, and simple grouping, using `@id` column references from the Croissant schema. Adjust field/column IDs as could be discovered in the prior cells. If no numeric field is available, the cell will report this gracefully.

In [ ]:
# EDA: Select and filter on a numeric field using its @id

import numpy as np
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

if selected_record_set_id is None:
    print('No record set with data available for EDA.')
else:
    df = dataframes[selected_record_set_id]
    # Identify numeric fields/columns by @id or by dtype
    numeric_field_id = None
    for col in df.columns:
        if np.issubdtype(df[col].dropna().astype('str').str.replace(',','').astype('float', errors='ignore').dtype, np.number):
            numeric_field_id = col
            break
    if numeric_field_id is None:
        print(f'No numeric field detected for record set {selected_record_set_id}.')
    else:
        print(f'Using numeric field {numeric_field_id} (@id) for EDA.')
        df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = df[numeric_field_id].mean() if not pd.isna(df[numeric_field_id].mean()) else 0
        print(f'Example filtering with threshold = mean = {threshold:.2f}')
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (using @id):")
        display(filtered_df.head())

        # Normalizing the numeric field
        filtered_df[f'{numeric_field_id}_normalized'] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized field {numeric_field_id} in filtered records:")
        display(filtered_df[[numeric_field_id, f'{numeric_field_id}_normalized']].head())

        # Group by another field if one exists
        group_field_id = None
        for col in df.columns:
            if col != numeric_field_id and df[col].nunique() > 1 and not np.issubdtype(df[col].dtype, np.number):
                group_field_id = col
                break

        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean {numeric_field_id} by {group_field_id} (both @id):")
            display(grouped_df.head())
        else:
            print('No suitable group field found for grouping.')

## 5. Visualization

Visualize distributions or relationships between fields. This code will use the selected numeric field (by `@id`) and any group field found previously, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style='whitegrid')

if selected_record_set_id is None or numeric_field_id is None:
    print('No data or numeric field found for visualization.')
else:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, color="skyblue")
    plt.xlabel(numeric_field_id + ' (@id)')
    plt.title(f'Distribution of numeric field ({numeric_field_id})')
    plt.show()

    if group_field_id:
        plt.figure(figsize=(8,5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.xlabel(group_field_id + ' (@id)')
        plt.ylabel(numeric_field_id + ' (@id)')
        plt.title(f'{numeric_field_id} by {group_field_id} (by @id)')
        plt.show()

## 6. Conclusion

This notebook demonstrated how to load, inspect, and process the FAIR² dataset using the mlcroissant library, referencing all entities explicitly by their `@id`. Further analysis can be extended by leveraging the Croissant model for rigorous and reproducible data science workflows.